# Multi-Hop Legal Information Retrieval
## Complete Kaggle Submission Pipeline

This notebook provides a complete, self-contained pipeline for:
1. Building a citation graph from court considerations and laws
2. TF-IDF based retrieval with multi-hop citation expansion
3. Generating the final submission file

## 1. Setup and Configuration

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import os
from pathlib import Path
from typing import List, Dict, Tuple, Set, Optional
from collections import defaultdict
from tqdm import tqdm

# Kaggle input paths
INPUT_PATH = '/kaggle/input/llm-agentic-legal-information-retrieval'
COURT_CONSIDERATIONS_PATH = os.path.join(INPUT_PATH, 'court_considerations.csv')
LAWS_PATH = os.path.join(INPUT_PATH, 'laws_de.csv')
TEST_PATH = os.path.join(INPUT_PATH, 'test.csv')

# Output path
OUTPUT_PATH = '/kaggle/working'
SUBMISSION_PATH = os.path.join(OUTPUT_PATH, 'submission.csv')

print("Kaggle Paths:")
print(f"  Input: {INPUT_PATH}")
print(f"  Output: {OUTPUT_PATH}")
print(f"\nFiles:")
print(f"  court_considerations.csv: {os.path.exists(COURT_CONSIDERATIONS_PATH)}")
print(f"  laws_de.csv: {os.path.exists(LAWS_PATH)}")
print(f"  test.csv: {os.path.exists(TEST_PATH)}")

## 2. Citation Extraction Functions

In [ ]:
def extract_bge_citations(text: str) -> Set[str]:
    """
    Extract BGE (Bundesgerichtsentscheid) citations from text.
    Patterns: BGE 139 I 2, BGE 138 I 189 E. 2.1, etc.
    """
    pattern = r'BGE\s+(\d+)\s+([IVX]+)\s+(\d+)'
    matches = re.findall(pattern, text)
    
    citations = set()
    for vol, section, page in matches:
        citations.add(f"BGE {vol} {section} {page}")
    
    return citations


def extract_article_citations(text: str) -> Set[str]:
    """
    Extract law article citations from text.
    Patterns: Art. 34 BV, Art. 95 lit. c BGG, etc.
    """
    citations = set()
    
    # Art. X LAW pattern
    art_pattern = r'Art\.?\s+(\d+)(?:\s+(?:Abs\.?\s+\d+|lit\.?\s+[a-z]))?\s+([A-Z]{2,}(?:/[A-Z]+)?)'
    for match in re.finditer(art_pattern, text):
        num, law = match.groups()
        citations.add(f"Art. {num} {law}")
    
    # Paragraph pattern
    para_pattern = r'§+\s*(\d+)(?:\s+(?:Abs\.?\s+\d+))?\s+([A-Z]{2,}(?:/[A-Z]+)?)'
    for match in re.finditer(para_pattern, text):
        num, law = match.groups()
        citations.add(f"§ {num} {law}")
    
    return citations


def normalize_citation_id(citation: str) -> str:
    """Normalize citation ID for matching"""
    bge_match = re.match(r'(BGE\s+\d+\s+[IVX]+\s+\d+)', citation)
    if bge_match:
        return bge_match.group(1)
    
    art_match = re.match(r'(Art\.?\s+\d+\s+(?:Abs\.?\s+\d+\s+)?[A-Z0-9]+)', citation)
    if art_match:
        return art_match.group(1)
    
    return citation

print("Citation extraction functions defined.")

## 3. Build Citation Graph

In [ ]:
def build_citation_graph(court_path: str, laws_path: str = None) -> List[Dict]:
    """
    Build citation graph by extracting references from document texts.
    """
    print("=" * 60)
    print("BUILDING CITATION GRAPH")
    print("=" * 60)
    
    # Load court considerations
    print(f"\n[1/4] Loading court considerations...")
    court_df = pd.read_csv(court_path)
    print(f"  Loaded {len(court_df):,} court documents")
    
    # Load laws if provided
    laws_df = None
    if laws_path and os.path.exists(laws_path):
        print(f"\n[2/4] Loading laws...")
        laws_df = pd.read_csv(laws_path)
        print(f"  Loaded {len(laws_df):,} law documents")
    
    # Build document index
    print("\n[3/4] Building document index...")
    doc_index = {}
    
    for _, row in court_df.iterrows():
        doc_id = row['citation']
        normalized = normalize_citation_id(doc_id)
        doc_index[doc_id] = doc_id
        if normalized != doc_id and normalized not in doc_index:
            doc_index[normalized] = doc_id
    
    if laws_df is not None:
        for _, row in laws_df.iterrows():
            doc_id = row['citation']
            normalized = normalize_citation_id(doc_id)
            doc_index[doc_id] = doc_id
            if normalized != doc_id and normalized not in doc_index:
                doc_index[normalized] = doc_id
    
    print(f"  Indexed {len(doc_index):,} document references")
    
    # Extract citations
    print("\n[4/4] Extracting citations...")
    documents = []
    total_citations = 0
    
    # Process court considerations
    for idx, row in tqdm(court_df.iterrows(), total=len(court_df), desc="Court docs"):
        doc_id = row['citation']
        text = str(row['text']) if pd.notna(row['text']) else ''
        
        # Extract and resolve citations
        all_cites = extract_bge_citations(text) | extract_article_citations(text)
        
        resolved = []
        for cite in all_cites:
            if cite in doc_index and doc_index[cite] != doc_id:
                resolved.append(doc_index[cite])
            else:
                normalized = normalize_citation_id(cite)
                if normalized in doc_index and doc_index[normalized] != doc_id:
                    resolved.append(doc_index[normalized])
        
        total_citations += len(resolved)
        documents.append({
            'id': doc_id,
            'text': text[:2000],  # Truncate for memory
            'citations': list(set(resolved))
        })
    
    # Process laws
    if laws_df is not None:
        for idx, row in tqdm(laws_df.iterrows(), total=len(laws_df), desc="Law docs"):
            doc_id = row['citation']
            text = str(row['text']) if pd.notna(row['text']) else ''
            
            all_cites = extract_bge_citations(text) | extract_article_citations(text)
            
            resolved = []
            for cite in all_cites:
                if cite in doc_index and doc_index[cite] != doc_id:
                    resolved.append(doc_index[cite])
                else:
                    normalized = normalize_citation_id(cite)
                    if normalized in doc_index and doc_index[normalized] != doc_id:
                        resolved.append(doc_index[normalized])
            
            total_citations += len(resolved)
            documents.append({
                'id': doc_id,
                'text': text[:2000],
                'citations': list(set(resolved))
            })
    
    docs_with_citations = sum(1 for d in documents if d['citations'])
    
    print(f"\n" + "=" * 60)
    print("CITATION GRAPH STATISTICS")
    print("=" * 60)
    print(f"  Total documents: {len(documents):,}")
    print(f"  Total citation links: {total_citations:,}")
    print(f"  Documents with citations: {docs_with_citations:,}")
    
    return documents

print("Citation graph builder defined.")

In [ ]:
# Build the citation graph
documents = build_citation_graph(COURT_CONSIDERATIONS_PATH, LAWS_PATH)

## 4. TF-IDF Retriever

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import time

class TFIDFRetriever:
    """
    Memory-efficient TF-IDF retriever with German stopwords.
    Optimized for faster initialization on Kaggle.
    """
    def __init__(self, documents: List[Dict]):
        self.documents = documents
        self.doc_ids = [doc['id'] for doc in documents]
        self.id_to_idx = {doc_id: idx for idx, doc_id in enumerate(self.doc_ids)}
        
        print(f"Initializing TF-IDF retriever for {len(documents):,} documents...")
        
        # German stopwords for legal text
        german_stopwords = [
            'der', 'die', 'das', 'und', 'in', 'zu', 'den', 'von',
            'mit', 'ist', 'auf', 'des', 'für', 'nicht', 'sich',
            'als', 'auch', 'es', 'an', 'werden', 'wird', 'sind',
            'bei', 'einer', 'eines', 'einem', 'einen', 'eine',
            'dem', 'dass', 'nach', 'zum', 'zur', 'aus', 'oder'
        ]
        
        self.vectorizer = TfidfVectorizer(
            max_features=20000,
            min_df=5,
            max_df=0.7,
            ngram_range=(1, 1),
            dtype=np.float32,
            sublinear_tf=True,
            norm='l2',
            stop_words=german_stopwords
        )
        
        # Step 1: Extract texts
        print("[1/3] Extracting document texts...")
        t0 = time.time()
        doc_texts = [doc.get('text', '') for doc in tqdm(documents, desc="Texts")]
        print(f"      Done in {time.time() - t0:.1f}s")
        
        # Step 2: Fit vectorizer (learn vocabulary)
        print("[2/3] Fitting vectorizer (building vocabulary)...")
        t0 = time.time()
        self.vectorizer.fit(doc_texts)
        print(f"      Vocabulary size: {len(self.vectorizer.vocabulary_):,}")
        print(f"      Done in {time.time() - t0:.1f}s")
        
        # Step 3: Transform documents
        print("[3/3] Transforming documents to TF-IDF matrix...")
        t0 = time.time()
        self.tfidf_matrix = self.vectorizer.transform(doc_texts)
        print(f"      Done in {time.time() - t0:.1f}s")
        
        print(f"\nMatrix shape: {self.tfidf_matrix.shape}")
        print(f"Memory: ~{self.tfidf_matrix.data.nbytes / 1024 / 1024:.1f} MB")
    
    def search(self, query: str, top_k: int = 20) -> List[Tuple[str, float]]:
        """Search and return (doc_id, score) pairs"""
        query_vec = self.vectorizer.transform([query])
        scores = (self.tfidf_matrix @ query_vec.T).toarray().flatten()
        
        top_indices = np.argpartition(scores, -min(top_k, len(scores)))[-top_k:]
        top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
        
        return [(self.doc_ids[i], float(scores[i])) for i in top_indices]

print("TF-IDF Retriever class defined.")

In [ ]:
import time
start = time.time()
retriever = TFIDFRetriever(documents)
print(f"Initialization took {time.time() - start:.1f}s")

## 5. Generate Submission

In [ ]:
# Load test queries
test_df = pd.read_csv(TEST_PATH)

# Handle column names
if 'query' in test_df.columns and 'query_text' not in test_df.columns:
    test_df['query_text'] = test_df['query']

print(f"Test queries: {len(test_df)}")
print(f"Columns: {test_df.columns.tolist()}")
test_df.head()

In [ ]:
print(f"Generating submission for {len(test_df)} queries...\n")

submission_rows = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Retrieval"):
    query_id = row['query_id']
    query_text = row['query_text']
    
    # Retrieve documents
    results = retriever.search(query_text, top_k=20)
    
    # Add to submission
    for rank, (doc_id, score) in enumerate(results, 1):
        submission_rows.append({
            'query_id': query_id,
            'doc_id': doc_id,
            'rank': rank
        })

# Create DataFrame
submission_df = pd.DataFrame(submission_rows)

# Remove duplicates (keep first = best rank)
submission_df = submission_df.drop_duplicates(subset=['query_id', 'doc_id'], keep='first')

print(f"\nGenerated {len(submission_df):,} submission rows")

In [ ]:
citation_map = {doc['id']: doc['citations'] for doc in documents}

print(f"Generating Multi-Hop submission in semicolon format...\n")

submission_data = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Retrieval"):
    query_id = row['query_id']
    query_text = row['query_text']

    initial_results = retriever.search(query_text, top_k=15)

    final_scores = {} # doc_id -> score

    for doc_id, score in initial_results:
        final_scores[doc_id] = max(final_scores.get(doc_id, 0), score)

        if doc_id in citation_map:
            for ref_id in citation_map[doc_id]:
                ref_score = score * 0.8
                final_scores[ref_id] = max(final_scores.get(ref_id, 0), ref_score)

    sorted_docs = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    top_docs = [doc[0] for doc in sorted_docs[:20]]

    gold_citations_str = ";".join(top_docs)

    submission_data.append({
        'query_id': query_id,
        'predicted_citations': gold_citations_str
    })

submission_df = pd.DataFrame(submission_data)

SUBMISSION_PATH = "/kaggle/working/submission.csv"
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"\nSubmission saved with format: query_id,gold_citations")
print(f"Total queries: {len(submission_df)}")
print("-" * 30)
print(submission_df.head(3))
